[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C56_Detection_Augmentation_Course/02_photometric/02_photometric_aug.ipynb)

# 02 · 光度增强与域鲁棒性（HSV 语义 / 大气散射 / 运动模糊 / 低光合成 / JPEG）

目标：把「光度增强」从一堆调用库的参数，变成**可以从物理量算出来的东西**。
重点是 TSR 特有的那条红线——**交通标志的颜色就是语义**——并把它量化成一个具体的度数。

本 notebook 你会亲手实现：
1. **RGB↔HSV** 的完整往返变换（纯 numpy），以及一个「颜色族」分类器；
2. **色相偏移多少度会让红色禁令牌落进黄色警告牌的区间**——精确到小数点后一位；
3. 逐通道**仿射算子**的色彩代数：证明亮度/对比度/雾对色相做了什么、对饱和度做了什么；
4. **大气散射雾化模型** `I = J·t + A(1-t)`，`t = exp(-βd)`，含深度图与能见度换算；
5. **运动模糊核**（方向 + 长度）与它那个**与焦距无关**的关键比值；
6. **卷帘快门**的行剪切与「行时间戳」量化；
7. **低光合成**：泊松散粒噪声 + 高斯读出噪声 + 数字增益 + 8-bit 量化，验证 SNR ∝ √N；
8. **JPEG 8×8 DCT 量化**，看压缩伪影如何吃掉限速牌上的数字。

> 心智模型：**光度增强不改一个标注，却可能改掉标签。
> 判断一个算子该不该用，先问「成像链路上哪一环产生它」，再问「它对色相做了什么」。**

## 1 · 合成一个 TSR 场景：颜色语义、深度图与 RGB↔HSV

先把工具备齐。场景是从 1920×1080 帧上裁下的一小块（1:1，不缩放），焦距 `f = 1000 px`，
所以「标志的像素直径 = f·S/Z」这条针孔关系是真的，可以直接代入真实距离。

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
np.set_printoptions(precision=4, suppress=True)

# ── 交通标志的「语义颜色」：GB 5768 / 维也纳公约 / MUTCD 共通的颜色族 ──
CLASS_RGB = {
    '禁令(红)': (0.80, 0.10, 0.12),
    '警告(黄)': (0.95, 0.78, 0.05),
    '指路(绿)': (0.05, 0.45, 0.22),
    '指示(蓝)': (0.05, 0.25, 0.65),
}

def rgb_to_hsv(rgb):
    '''RGB[0,1] -> HSV，H∈[0,360)，S,V∈[0,1]。支持任意形状（最后一维为 3）。'''
    a = np.asarray(rgb, dtype=float); shp = a.shape
    x = a.reshape(-1, 3)
    r, g, b = x[:, 0], x[:, 1], x[:, 2]
    mx, mn = x.max(1), x.min(1)
    d = mx - mn
    h = np.zeros_like(mx)
    m = d > 1e-12
    i = m & (mx == r); h[i] = ((g[i] - b[i]) / d[i]) % 6.0
    i = m & (mx == g); h[i] = (b[i] - r[i]) / d[i] + 2.0
    i = m & (mx == b); h[i] = (r[i] - g[i]) / d[i] + 4.0
    h = (h * 60.0) % 360.0
    s = np.where(mx > 1e-12, d / np.where(mx > 1e-12, mx, 1.0), 0.0)
    return np.stack([h, s, mx], 1).reshape(shp)

def hsv_to_rgb(hsv):
    a = np.asarray(hsv, dtype=float); shp = a.shape
    x = a.reshape(-1, 3)
    h = x[:, 0] % 360.0
    s = np.clip(x[:, 1], 0, 1)
    v = np.clip(x[:, 2], 0, 1)
    c = v * s
    hp = h / 60.0
    xx = c * (1.0 - np.abs(hp % 2.0 - 1.0))
    z = np.zeros_like(c)
    opts = np.stack([np.stack([c, xx, z], 1), np.stack([xx, c, z], 1),
                     np.stack([z, c, xx], 1), np.stack([z, xx, c], 1),
                     np.stack([xx, z, c], 1), np.stack([c, z, xx], 1)], 0)   # (6, N, 3)
    idx = np.floor(hp).astype(int) % 6
    out = opts[idx, np.arange(len(idx))]
    return np.clip(out + (v - c)[:, None], 0, 1).reshape(shp)

# 往返一致性（这是后面所有色彩分析的地基）
_probe = rng.random((200, 3))
assert np.allclose(hsv_to_rgb(rgb_to_hsv(_probe)), _probe, atol=1e-12), 'HSV 往返不闭合'

CLASS_HUE = {k: float(rgb_to_hsv(np.array(v))[0]) for k, v in CLASS_RGB.items()}
print('颜色族色相：')
for k, v in CLASS_HUE.items():
    s_, v_ = rgb_to_hsv(np.array(CLASS_RGB[k]))[1:]
    print(f'  {k}   H={v:7.2f}°   S={s_:.3f}  V={v_:.3f}')
assert abs(CLASS_HUE['禁令(红)'] - 358.29) < 0.05
assert abs(CLASS_HUE['指示(蓝)'] - 220.00) < 0.05
print('✅ RGB↔HSV 就位；颜色族色相已标定')

In [ ]:
# ── 合成场景：天空 / 路面 / 一块圆形标志 / 杆件 / 深度图 ──
IMG_H, IMG_W = 96, 128          # 从 1920x1080 帧上裁下的一块（1:1，无缩放）
F_PX = 1000.0                   # 相机焦距（像素）—— 真实数量级
Y_HORIZON = 46                  # 地平线所在行

def make_scene(cls='禁令(红)', Z=30.0, S_sign=0.60, seed=0):
    r = np.random.default_rng(seed)
    yy = np.arange(IMG_H)[:, None, None]
    sky  = np.array([0.62, 0.72, 0.88]).reshape(1, 1, 3)
    road = np.array([0.30, 0.29, 0.28]).reshape(1, 1, 3)
    img = np.where(yy > Y_HORIZON, road, sky) * np.ones((IMG_H, IMG_W, 1))
    img = img + r.normal(0, 0.012, img.shape)                    # 轻微纹理

    # 深度图：天空 200 m；路面按行从 120 m 线性递减到 12 m（简化但保留「近下远上」）
    prof = np.interp(np.arange(IMG_H), [Y_HORIZON, IMG_H - 1], [120.0, 12.0])
    depth = np.where(np.arange(IMG_H) > Y_HORIZON, prof, 200.0)[:, None] * np.ones((IMG_H, IMG_W))

    # 标志：针孔关系 d_px = f·S/Z
    d_px = F_PX * S_sign / Z
    R = d_px / 2.0
    cy, cx = 30.0, 64.0
    Y, X = np.mgrid[0:IMG_H, 0:IMG_W]
    rr = np.hypot(Y - cy, X - cx)
    mask = rr <= R
    rim  = mask & (rr > 0.74 * R)
    core = rr <= 0.74 * R
    img[rim] = CLASS_RGB[cls]
    img[core] = (0.93, 0.93, 0.92)
    bar = core & (np.abs(Y - cy) <= max(1.0, 0.16 * R))          # 牌面上的深色图案
    img[bar] = (0.12, 0.12, 0.13)
    depth[mask] = Z
    pole = (np.abs(X - cx) <= 1) & (Y > cy + R) & (Y < Y_HORIZON + 6)
    img[pole] = (0.45, 0.45, 0.46)
    depth[pole] = Z

    img = np.clip(img, 0, 1)
    ys, xs = np.where(mask)
    bbox = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
    return dict(img=img, depth=depth, mask=mask, rim=rim, bbox=bbox, Z=Z,
                cls=cls, d_px=d_px, y_h=Y_HORIZON)

def sign_hue(scene, img=None):
    '''量圈边（承载颜色语义的那一环）的平均色相/饱和度/明度。'''
    im = scene['img'] if img is None else img
    return rgb_to_hsv(im[scene['rim']].mean(0))

sc = make_scene('禁令(红)', Z=30.0)
h0, s0, v0 = sign_hue(sc)
print(f"场景：{sc['cls']}  Z={sc['Z']:.0f} m  →  标志直径 {sc['d_px']:.1f} px  bbox={sc['bbox']}")
print(f"圈边颜色：H={h0:.2f}°  S={s0:.3f}  V={v0:.3f}")
print(f"深度图范围：{sc['depth'].min():.1f} – {sc['depth'].max():.1f} m（标志处 {sc['Z']:.0f} m）")
assert abs(sc['d_px'] - 20.0) < 1e-9, 'f·S/Z = 1000×0.6/30 = 20 px'
assert abs(h0 - CLASS_HUE['禁令(红)']) < 1.0, '圈边应保持红色族色相'
assert sc['bbox'][3] <= sc['y_h'], '路侧标志的框应完全在地平线以上'
print('✅ 场景就位：20 px 的标志 = 1000 px 焦距下 30 m 外的 0.6 m 圆牌')

## 2 · 光度算子的色彩代数：仿射安全、加性掉饱和、gamma 微移色相

常见光度算子几乎全是**逐通道仿射** `I' = a·I + b`。对这一族可以精确说清它做了什么：

- 通道差被同乘 `a` → `(g-b)/Δ` 这类比值不变 → **色相严格不变**（前提 b 三通道相同）；
- `S = Δ/max = a·Δ_J / (a·max_J + b)` → **`b > 0` 时饱和度必然下降**。

而 `gamma` 是非线性的，它会轻微移动色相。下面把这些都量出来。

In [ ]:
def op_brightness_mul(img, k):   return np.clip(img * k, 0.0, 1.0)
def op_brightness_add(img, b):   return np.clip(img + b, 0.0, 1.0)
def op_contrast(img, c):
    mu = float(img.mean())
    return np.clip((img - mu) * c + mu, 0.0, 1.0)
def op_gamma(img, g):            return np.clip(img, 0.0, 1.0) ** g
def op_hsv_jitter(img, dh=0.0, ds=1.0, dv=1.0):
    hsv = rgb_to_hsv(img)
    hsv[..., 0] = (hsv[..., 0] + dh) % 360.0
    hsv[..., 1] = np.clip(hsv[..., 1] * ds, 0.0, 1.0)
    hsv[..., 2] = np.clip(hsv[..., 2] * dv, 0.0, 1.0)
    return hsv_to_rgb(hsv)

# 恒等性检查：HSV 抖动的零参数必须是恒等变换
assert np.allclose(op_hsv_jitter(sc['img'], 0.0, 1.0, 1.0), sc['img'], atol=1e-12)

def hue_shift(a, b):
    d = abs(a - b) % 360.0
    return min(d, 360.0 - d)

base = np.array(CLASS_RGB['禁令(红)'])
rows = []
for name, fn in [('乘性亮度 ×1.25', lambda x: op_brightness_mul(x, 1.25)),
                 ('乘性亮度 ×0.70', lambda x: op_brightness_mul(x, 0.70)),
                 ('加性亮度 +0.20', lambda x: op_brightness_add(x, 0.20)),
                 ('对比度 ×0.60',   lambda x: np.clip((x - 0.45) * 0.6 + 0.45, 0, 1)),
                 ('gamma 0.50',     lambda x: op_gamma(x, 0.50)),
                 ('gamma 2.20',     lambda x: op_gamma(x, 2.20)),
                 ('饱和度 ×0.50',   lambda x: op_hsv_jitter(x, 0.0, 0.5, 1.0)),
                 ('色相 +30°',      lambda x: op_hsv_jitter(x, 30.0, 1.0, 1.0))]:
    out = fn(base)
    h, s, v = rgb_to_hsv(out)
    rows.append((name, hue_shift(h, CLASS_HUE['禁令(红)']), s / 0.875, v))

print(f"{'算子':<16s} {'|Δ色相|':>9s} {'饱和度比':>10s} {'明度':>8s}   判定")
for name, dh, sr, v in rows:
    verdict = '✅ 色相不变' if dh < 0.01 else ('⚠️ 轻微移色' if dh < 5 else '❌ 破坏颜色语义')
    print(f'{name:<16s} {dh:>9.3f}° {sr:>10.3f} {v:>8.3f}   {verdict}')

# 仿射类算子：色相严格不变
for fn in [lambda x: op_brightness_mul(x, 1.25), lambda x: op_brightness_add(x, 0.20),
           lambda x: np.clip((x - 0.45) * 0.6 + 0.45, 0, 1)]:
    assert hue_shift(rgb_to_hsv(fn(base))[0], CLASS_HUE['禁令(红)']) < 1e-9
# 加性亮度 = 饱和度衰减器
assert rgb_to_hsv(op_brightness_add(base, 0.20))[1] < 0.875 - 0.15
# 乘性亮度 = 纯 V 缩放，饱和度也不变
assert abs(rgb_to_hsv(op_brightness_mul(base, 0.70))[1] - 0.875) < 1e-12
# gamma：非线性，色相有微小偏移但远小于安全余量
assert 0.1 < hue_shift(rgb_to_hsv(op_gamma(base, 2.2))[0], CLASS_HUE['禁令(红)']) < 3.0
print()
print('✅ 结论一：乘性亮度是最安全的算子（色相与饱和度都严格不变）')
print('✅ 结论二：加性亮度/对比度/雾都是仿射 → 色相不变，但**饱和度必然下降**')
print('✅ 结论三：gamma 会移色相，但红牌只移约 1.4°，远在 25° 的红线以内')
print('❌ 结论四：显式色相抖动是唯一一个能直接跨越类别边界的算子')

## 3 · 招牌实验：色相偏移多少度，红色禁令牌会变成黄色警告牌

用「最近色相」作为颜色族分类器，扫描色相偏移，找出**翻类的临界角**。

In [ ]:
def hue_dist(a, b):
    d = np.abs((np.asarray(a, dtype=float) - b) % 360.0)
    return np.minimum(d, 360.0 - d)

def classify_by_hue(h, class_hue=CLASS_HUE):
    names = list(class_hue)
    d = np.array([hue_dist(h, class_hue[n]) for n in names])
    return names[int(np.argmin(d))]

def flip_angle(cls, sign, class_hue=CLASS_HUE, step=0.1, limit=180.0):
    '''沿 sign(±1) 方向偏移色相，返回第一个翻类的角度（度）。'''
    h0 = class_hue[cls]
    for d in np.arange(step, limit + step / 2, step):
        if classify_by_hue((h0 + sign * d) % 360.0, class_hue) != cls:
            return float(d), classify_by_hue((h0 + sign * d) % 360.0, class_hue)
    return float('inf'), None

print(f"{'颜色族':<12s} {'h_c':>8s} {'负向余量':>10s} {'翻成':<12s} {'正向余量':>10s} {'翻成'}")
margins = {}
for cls in CLASS_HUE:
    dn, tn = flip_angle(cls, -1)
    dp, tp = flip_angle(cls, +1)
    margins[cls] = (dn, dp)
    print(f'{cls:<12s} {CLASS_HUE[cls]:>7.2f}° {dn:>9.1f}° {tn:<12s} {dp:>9.1f}° {tp}')

MIN_MARGIN = min(min(v) for v in margins.values())
print()
print(f'★ 全局最小余量 = {MIN_MARGIN:.1f}°，出现在「红↔黄」这一对上')
print('  —— 这恰好是 TSR 里后果最严重的一对（禁令 vs 警告，下游动作完全不同）')

dp_red, tp_red = flip_angle('禁令(红)', +1)
dn_red, tn_red = flip_angle('禁令(红)', -1)
assert 24.5 <= dp_red <= 26.0 and tp_red == '警告(黄)', (dp_red, tp_red)
assert 68.0 <= dn_red <= 70.5 and tn_red == '指示(蓝)', (dn_red, tn_red)
assert 24.5 <= MIN_MARGIN <= 26.0
print()
print(f'✅ 红色禁令牌只要色相 **+{dp_red:.1f}°** 就落进黄色警告牌的色相区间；')
print(f'   反方向要 -{dn_red:.1f}° 才碰到蓝色 —— **风险是高度不对称的**。')

In [ ]:
# ── 把常用库的默认值放进同一把尺子 ──
# 注意：OpenCV 的 HSV 色相通道是 0–179（1 单位 = 2°），这是最常被漏掉的换算
LIB_DEFAULTS = [
    ('YOLOv5/v8  hsv_h=0.015（乘性）',                  5.4,  '乘性变换，纯红处偏移≈0'),
    ('torchvision ColorJitter(hue=0.1)',               36.0, 'hue 参数单位是「圈」，0.1 圈 = 36°'),
    ('albumentations HueSaturationValue(20)',          40.0, 'OpenCV 单位 ×2 = 40°'),
    ('RandAugment Color/Hue（大 magnitude）',          60.0, '搜索目标里没有颜色语义约束'),
    ('本课建议：0.5 × 最小余量',        0.5 * MIN_MARGIN,   '留一半余量给传感器白平衡偏差'),
]
print(f"{'配置':<42s} {'最大偏移':>9s} {'相对余量':>9s}  判定")
for name, amp, note in LIB_DEFAULTS:
    ratio = amp / MIN_MARGIN
    verdict = '✅ 安全' if ratio <= 0.75 else ('⚠️ 逼近红线' if ratio < 1.0 else '❌ 越界')
    print(f'{name:<42s} {amp:>8.1f}° {ratio:>8.2f}×  {verdict}   {note}')

# 直接验证：用 albumentations 的默认幅度抖一张红牌，看它被判成什么
img_red = make_scene('禁令(红)', Z=30.0)['img']
sc_red = make_scene('禁令(红)', Z=30.0)
bad = op_hsv_jitter(img_red, dh=40.0)
good = op_hsv_jitter(img_red, dh=0.5 * MIN_MARGIN)
h_bad = sign_hue(sc_red, bad)[0]
h_good = sign_hue(sc_red, good)[0]
print()
print(f'原图圈边  H={sign_hue(sc_red)[0]:7.2f}°  → 判为 {classify_by_hue(sign_hue(sc_red)[0])}')
print(f'+40°  抖动 H={h_bad:7.2f}°  → 判为 {classify_by_hue(h_bad)}   ← **标签已经被改掉了**')
print(f'+{0.5*MIN_MARGIN:.1f}° 抖动 H={h_good:7.2f}°  → 判为 {classify_by_hue(h_good)}')
assert classify_by_hue(h_bad) != '禁令(红)', 'albumentations 默认幅度应当翻类'
assert classify_by_hue(h_good) == '禁令(红)', '建议幅度必须保住类别'
print()
print('⚠️  注意 YOLOv5 的默认值为什么反而安全：它做的是**乘性**色相变换 H\' = H·r，')
print('    偏移正比于色相值本身，而纯红在 OpenCV 里色相值接近 0 或 179 —— ')
print('    **红色恰好是这个实现下最被保护的颜色。这是幸运，不是设计。**')

## 4 · 大气散射雾化：`I = J·t + A(1-t)`，`t = exp(-βd)`

雾必须依赖**深度**。同时验证前面的色彩代数：雾是仿射变换，所以
**色相的偏移完全来自大气光 A 的非中性**（天空偏蓝）。

In [ ]:
def beta_from_visibility(V):
    '''Koschmieder：能见度定义为对比度衰减到 5% 的距离 → β = -ln(0.05)/V ≈ 3.0/V'''
    return -np.log(0.05) / float(V)

def transmittance(d, V):
    return np.exp(-beta_from_visibility(V) * np.asarray(d, dtype=float))

def apply_fog(img, depth, V, A=(0.85, 0.86, 0.90)):
    t = transmittance(depth, V)[..., None]
    return np.clip(img * t + np.array(A).reshape(1, 1, 3) * (1.0 - t), 0.0, 1.0)

sc = make_scene('禁令(红)', Z=30.0)
V_TEST = 100.0
fog = apply_fog(sc['img'], sc['depth'], V_TEST)
t_sign = float(transmittance(sc['Z'], V_TEST))

# ① 对比度被严格乘以 t（同深度处，A 是常数 → 差值项只剩 t）
bg_ring = (~sc['mask']) & (np.arange(IMG_H)[:, None] < sc['y_h'])   # 标志周围的天空
c_clear = float(np.abs(sc['img'][sc['rim']].mean(0) - sc['img'][bg_ring].mean(0)).mean())
c_fog   = float(np.abs(fog[sc['rim']].mean(0)     - fog[bg_ring].mean(0)).mean())
print(f'能见度 V={V_TEST:.0f} m  →  β={beta_from_visibility(V_TEST):.4f} /m，标志处 t={t_sign:.4f}')
print(f'① 对比度：清晰 {c_clear:.4f} → 雾中 {c_fog:.4f}   比值 {c_fog/c_clear:.4f}（理论应为 t_sign）')

# ② / ③ 饱和度与色相
h_c, s_c, v_c = sign_hue(sc)
h_f, s_f, v_f = sign_hue(sc, fog)
print(f'② 饱和度：{s_c:.3f} → {s_f:.3f}   比值 {s_f/s_c:.3f}   ← **雾是饱和度杀手**')
print(f'③ 色相  ：{h_c:.2f}° → {h_f:.2f}°   偏移 {hue_shift(h_f, h_c):.2f}°   ← 远小于 25° 红线')

# 关键验证：A 换成中性灰，色相应当**严格不变**
fog_neutral = apply_fog(sc['img'], sc['depth'], V_TEST, A=(0.87, 0.87, 0.87))
h_n = sign_hue(sc, fog_neutral)[0]
print(f'   把 A 换成中性灰(0.87,0.87,0.87)：色相偏移 {hue_shift(h_n, h_c):.6f}°')

assert abs(c_fog / c_clear - t_sign) < 0.02, '对比度衰减必须等于透射率'
assert s_f / s_c < 0.55, '雾必须显著压低饱和度'
assert hue_shift(h_f, h_c) < 15.0, '偏蓝的大气光只带来小幅色相偏移'
assert hue_shift(h_n, h_c) < 1e-9, '中性大气光下色相必须严格不变'
assert classify_by_hue(h_f) == '禁令(红)', '雾不应改变颜色族判定'
print()
print('✅ 三条结论全部验证：**雾杀对比度和饱和度，几乎不动色相**。')
print('   推论：靠饱和度做决策的分类器雾天直接失效，靠色相的相对稳健 ——')
print('   **前提是你训练时没把色相抖乱。**（这两节的结论是连在一起的）')

In [ ]:
# ── 能见度 → 最远可用距离：一个应该被背下来的表 ──
def max_range_at_contrast(V, t_min):
    '''保住 t_min 比例的原始对比度，最远能到多少米。'''
    return float(np.log(1.0 / t_min) / beta_from_visibility(V))

CRUISE = 25.0     # m/s，约 90 km/h
print(f"{'能见度 V':>9s} {'β (/m)':>9s} {'t@30m':>8s} {'t@60m':>9s} "
      f"{'保住20%对比度的最远距离':>22s} {'剩余时间':>9s}")
for V in [500, 200, 100, 50, 20]:
    b = beta_from_visibility(V)
    dmax = max_range_at_contrast(V, 0.20)
    print(f'{V:>8.0f} m {b:>9.4f} {float(transmittance(30, V)):>8.3f} '
          f'{float(transmittance(60, V)):>9.4f} {dmax:>21.0f} m {dmax/CRUISE:>8.2f} s')

d50 = max_range_at_contrast(50, 0.20)
assert 26.0 < d50 < 28.0, d50
assert abs(float(transmittance(60, 50)) - 0.0275) < 0.002
print()
print(f'★ 能见度 50 m 时，标志最远只能在 {d50:.0f} m 处保住 20% 对比度；')
print(f'  以 90 km/h 巡航，从检出到经过只剩 {d50/CRUISE:.1f} 秒 ——')
print('  **多帧确认 + 决策 + 执行全部要挤进这 1 秒。**')
print('  所以雾天 TSR 不是「精度掉一点」，而是**系统性地来不及**；')
print('  工程解法是降速/降级/依赖地图先验，而不是指望模型再强一点。')

## 5 · 运动模糊核与那个「与焦距无关」的比值

`L_blur = f·X·v/Z² · t_exp`，`s_px = f·S/Z`  ⟹  `L/s = X·v·t_exp/(S·Z)`——**f 被消掉了**。

In [ ]:
def conv2d_same(img, k):
    '''边缘复制填充的 same 卷积（纯 numpy，支持 HxW 与 HxWxC）。'''
    a = img[..., None] if img.ndim == 2 else img
    kh, kw = k.shape
    assert kh % 2 == 1 and kw % 2 == 1, '核尺寸必须是奇数'
    ph, pw = kh // 2, kw // 2
    pad = np.pad(a, ((ph, ph), (pw, pw), (0, 0)), mode='edge')
    out = np.zeros_like(a, dtype=float)
    H, W = a.shape[:2]
    for i in range(kh):
        for j in range(kw):
            if k[i, j] != 0.0:
                out += k[i, j] * pad[i:i + H, j:j + W]
    return out[..., 0] if img.ndim == 2 else out

def motion_kernel(length, angle_deg):
    '''长度 length（像素）、方向 angle_deg 的线状运动模糊核。'''
    L = max(1, int(round(length)))
    ks = 2 * int(np.ceil(L / 2)) + 1
    k = np.zeros((ks, ks))
    c = ks // 2
    th = np.deg2rad(angle_deg)
    for i in range(L):
        s = i - (L - 1) / 2.0
        y = int(round(c - s * np.sin(th)))
        x = int(round(c + s * np.cos(th)))
        k[y, x] += 1.0
    return k / k.sum()

k5 = motion_kernel(5, 0.0)
assert abs(k5.sum() - 1.0) < 1e-12, '核必须归一化（否则会改变整体亮度）'
flat = np.full((20, 20), 0.4)
assert np.allclose(conv2d_same(flat, motion_kernel(7, 30.0)), 0.4, atol=1e-12), '常量图卷积后应不变'
print('运动模糊核 length=5, angle=0：')
print(k5[k5.shape[0] // 2 - 1: k5.shape[0] // 2 + 2])

def blur_len_px(f, X, Z, v, t_exp):   return f * X * v / Z ** 2 * t_exp
def sign_px(f, S, Z):                 return f * S / Z

X_OFF, S_SIGN = 3.0, 0.60      # 路侧标志的横向偏移 / 物理直径
print()
print(f"{'场景':<18s} {'v(m/s)':>7s} {'Z(m)':>6s} {'t_exp':>8s} "
      f"{'L(px)':>7s} {'s(px)':>7s} {'L/s':>8s}  后果")
CASES = [('高速白天巡航', 25, 30, 0.010, '可忽略'),
         ('高速白天近距', 25, 10, 0.010, '边缘变软'),
         ('夜间城区',     14, 10, 0.030, '数字糊'),
         ('夜间高速近距', 25, 10, 0.030, '**高频被抹掉**'),
         ('雨夜长曝光',   20,  8, 0.040, '只剩颜色与轮廓')]
for name, v, Z, te, note in CASES:
    L = blur_len_px(F_PX, X_OFF, Z, v, te)
    s = sign_px(F_PX, S_SIGN, Z)
    print(f'{name:<18s} {v:>7.0f} {Z:>6.0f} {te*1000:>6.0f} ms {L:>7.2f} {s:>7.1f} '
          f'{L/s:>7.1%}  {note}')

# 关键性质：比值与焦距无关
for f_try in [500.0, 1000.0, 2500.0]:
    r_ = blur_len_px(f_try, X_OFF, 10.0, 25.0, 0.030) / sign_px(f_try, S_SIGN, 10.0)
    assert abs(r_ - X_OFF * 25.0 * 0.030 / (S_SIGN * 10.0)) < 1e-12
print()
print('✅ **L/s 与焦距无关** —— 换长焦看远处标志，模糊和目标一起放大，比值不变。')
print('   所以「上长焦」解决不了运动模糊，这是一个第一反应常想错的结论。')

# 真的把它糊一遍，看牌面高频损失了多少
sc10 = make_scene('禁令(红)', Z=10.0)          # 60 px 的标志
L_night = blur_len_px(F_PX, X_OFF, 10.0, 25.0, 0.030)
blurred = conv2d_same(sc10['img'], motion_kernel(L_night, 5.0))
def hf_energy(im, box, axis=None):
    # axis=1 只看水平方向高频（运动模糊近水平时应当看这个方向）；axis=None 看各向同性
    x0, y0, x1, y1 = box
    p = im[y0:y1, x0:x1].mean(-1)
    gv = float(np.abs(np.diff(p, axis=0)).mean())
    gh = float(np.abs(np.diff(p, axis=1)).mean())
    return gh if axis == 1 else (gv if axis == 0 else (gv + gh) / 2)

e0, e1 = hf_energy(sc10['img'], sc10['bbox'], 1), hf_energy(blurred, sc10['bbox'], 1)
print(f'\n夜间高速近距（L={L_night:.1f} px，标志 {sc10["d_px"]:.0f} px，模糊方向近水平）：')
print(f'  牌面**水平**高频能量 {e0:.4f} → {e1:.4f}，保留 {e1/e0:.1%}')
print(f'  垂直方向高频保留 '
      f'{hf_energy(blurred, sc10["bbox"], 0)/hf_energy(sc10["img"], sc10["bbox"], 0):.1%}'
      f'  ← 运动模糊是**各向异性**的，只削与运动方向平行的那一半信息')
assert e1 < e0 * 0.65, '这么长的水平模糊必须显著削掉水平高频'
assert (hf_energy(blurred, sc10['bbox'], 0) / hf_energy(sc10['img'], sc10['bbox'], 0)
        > e1 / e0), '垂直方向受损应当小于水平方向'
print('  ⚠️ 这正是「夜间检得出但认错」的物理根因：区分 60/80 的那点高频没了。')

## 6 · 卷帘快门：畸变可忽略，时间戳不可忽略

In [ ]:
def rolling_shutter(img, total_shift_px):
    '''逐行横向剪切：第 r 行相对第 0 行平移 total_shift_px·r/(H-1)（线性插值）。'''
    H, W = img.shape[:2]
    xs = np.arange(W, dtype=float)
    out = np.empty_like(img)
    for r in range(H):
        s = total_shift_px * r / (H - 1)
        for c in range(img.shape[2]):
            out[r, :, c] = np.interp(xs - s, xs, img[r, :, c])
    return out

T_RO, H_FULL, YAW = 0.020, 1080, 0.5        # 整帧读出 20 ms / 全帧高 1080 / 横摆 0.5 rad/s
frame_shift = F_PX * YAW * T_RO             # 整帧横向错切（像素）
sign_rows = sc['bbox'][3] - sc['bbox'][1]
obj_shift = frame_shift * sign_rows / H_FULL
ego_move = 30.0 * T_RO                      # 30 m/s 巡航下，帧内首尾行的自车位移

print(f'整帧读出时间 T_ro = {T_RO*1000:.0f} ms，横摆 ω = {YAW} rad/s')
print(f'  · 整帧横向错切 = f·ω·T_ro = {frame_shift:.1f} px')
print(f'  · 一块 {sign_rows} 行高的标志，其**内部**错切 = {obj_shift:.3f} px  ← 可忽略')
print(f'  · 帧内首尾行的时间差 = {T_RO*1000:.0f} ms → 30 m/s 下自车位移 {ego_move:.2f} m')
assert obj_shift < 1.0, '小目标的帧内卷帘畸变小于 1 px'
assert ego_move > 0.5, '整帧时间差对应的自车位移是米级的'

# 投影误差：0.6 m 的位姿误差，投到 30 m 外的标志上是多少像素？
lateral_err_px = F_PX * ego_move / 30.0 * 0.15   # 取 15% 分量投到像平面（保守估计）
print(f'  · 若用「帧时间戳」做运动补偿，30 m 外目标的预测位置误差可达 ~{lateral_err_px:.1f} px')
print(f'    对一个 {sc["d_px"]:.0f} px 的目标，这足以让 IoU 关联失败 → 跟踪 ID 频繁切换')
assert lateral_err_px > 2.0

rs = rolling_shutter(sc['img'], frame_shift)
print(f'\n实际剪切后，图像与原图的平均绝对差 = {np.abs(rs - sc["img"]).mean():.5f}')
print(f'标志框内的平均绝对差 = '
      f'{np.abs(rs - sc["img"])[sc["bbox"][1]:sc["bbox"][3], sc["bbox"][0]:sc["bbox"][2]].mean():.5f}')
print()
print('✅ 结论：**卷帘畸变不用做成增强**（对小目标像素分布几乎没影响），')
print('   但**时序融合必须用行时间戳**（C55 模块 04 的运动补偿依赖它）。')
print('⚠️  症状指纹：跟踪 ID 频繁切换、迟滞状态机反复重置 ——')
print('    根因在传感器时间模型上，在跟踪算法里怎么调都调不好。')

## 7 · 低光合成：泊松散粒 + 高斯读出 + 数字增益 + 量化

物理上正确的做法：**sRGB 反推回线性 → 缩放曝光 → 泊松采样 → 加读出噪声 →
数字增益拉回亮度 → 重新 gamma 编码 → 量化到 8 bit**。
`img + N(0,σ)` 这一行代码在物理上是错的。

In [ ]:
def low_light(img, exposure_ratio, full_well=8000.0, read_noise_e=3.0,
              gamma=2.2, bits=8, seed=0):
    r = np.random.default_rng(seed)
    lin = np.clip(img, 0.0, 1.0) ** gamma                 # sRGB → 线性辐射（近似）
    N = lin * full_well * exposure_ratio                  # 期望光电子数
    sig = r.poisson(N).astype(float) + r.normal(0.0, read_noise_e, N.shape)
    out = sig / (full_well * exposure_ratio)              # 数字增益 = 1/曝光比（把亮度拉回来）
    out = np.clip(out, 0.0, 1.0) ** (1.0 / gamma)
    q = 2 ** bits - 1
    return np.round(out * q) / q

def patch_snr(im):
    p = im.mean(-1)
    return float(p.mean() / (p.std() + 1e-12))

# 用一块**完全均匀**的灰板量 SNR：场景里的纹理会给 SNR 加一个与曝光无关的地板，
# 把 √N 的规律掩盖掉 —— 这本身就是「测噪声要用平场（flat field）」的工程常识。
FLAT = np.full((48, 48, 3), 0.35)
print(f"{'曝光比':>8s} {'平场SNR':>10s} {'相对 1.0':>10s} {'理论 √比值':>12s}")
snrs = {}
for e in [1.0, 1/2, 1/4, 1/8, 1/16, 1/64]:
    snrs[e] = patch_snr(low_light(FLAT, e, seed=3))
    print(f'{e:>8.4f} {snrs[e]:>10.2f} {snrs[e]/snrs[1.0]:>10.3f} {np.sqrt(e):>12.3f}')

ratio = snrs[1.0] / snrs[1/16]
print(f'\n曝光降 16× → SNR 降 {ratio:.2f}×（散粒噪声主导时理论值 √16 = 4）')
assert 3.0 <= ratio <= 5.5, ratio
assert all(snrs[a] > snrs[b] for a, b in [(1.0, 1/4), (1/4, 1/16), (1/16, 1/64)]), 'SNR 必须单调下降'

# 低光对牌面的破坏：用「与原图的偏差」度量，不要用「梯度能量」
# —— 噪声会**凭空造出**梯度，梯度能量反而上升，这个陷阱与 JPEG 那一节是同一个
x0, y0, x1, y1 = sc['bbox']
ref_patch = sc['img'][y0:y1, x0:x1]
print()
print(f"{'曝光比':>8s} {'牌面RMSE':>10s} {'牌面PSNR':>10s} {'梯度能量比':>11s} {'圈边色相偏移':>13s}")
for e in [1/4, 1/16, 1/64]:
    im = low_light(sc['img'], e, seed=5)
    rmse = float(np.sqrt(np.mean((im[y0:y1, x0:x1] - ref_patch) ** 2)))
    print(f'{e:>8.4f} {rmse:>10.4f} {10*np.log10(1/max(rmse**2,1e-12)):>9.2f} dB '
          f'{hf_energy(im, sc["bbox"])/hf_energy(sc["img"], sc["bbox"]):>10.1%} '
          f'{hue_shift(sign_hue(sc, im)[0], sign_hue(sc)[0]):>12.2f}°')
print('  ↑ 注意「梯度能量比」超过 100% —— **噪声凭空造出了高频**。')
print('    所以度量退化必须看「与原图的偏差」，不能看「梯度能量的绝对值」。')
print()
print('✅ SNR ∝ √N：曝光降 16 倍，信噪比只降 4 倍 ——')
print('   这解释了「稍微暗一点还行，暗过某个点突然崩」：')
print('   一旦光电子数 N 掉到与读出噪声 σ_r² 同量级，读出噪声接管，SNR 转为线性下降。')
print('⚠️  夜间的两条路都在削同一样东西：')
print('    拉长曝光 → 运动模糊按 t_exp 线性增长（第 5 节）')
print('    拉高增益 → SNR 按 √N 下降（本节）')
print('    **所以低光合成与运动模糊必须成对进增强配方。**')

## 8 · JPEG 8×8 DCT 量化：压缩伪影如何吃掉限速牌上的数字

In [ ]:
def dct_mat(N=8):
    n = np.arange(N)
    M = np.cos(np.pi * (2 * n[None, :] + 1) * n[:, None] / (2 * N)) * np.sqrt(2.0 / N)
    M[0] = M[0] / np.sqrt(2.0)
    return M

DCT8 = dct_mat(8)
assert np.allclose(DCT8 @ DCT8.T, np.eye(8), atol=1e-12), 'DCT 基必须正交归一'

Q50 = np.array([
    [16, 11, 10, 16,  24,  40,  51,  61],
    [12, 12, 14, 19,  26,  58,  60,  55],
    [14, 13, 16, 24,  40,  57,  69,  56],
    [14, 17, 22, 29,  51,  87,  80,  62],
    [18, 22, 37, 56,  68, 109, 103,  77],
    [24, 35, 55, 64,  81, 104, 113,  92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103,  99]], dtype=float)

def qtable(quality):
    s = 5000.0 / quality if quality < 50 else 200.0 - 2.0 * quality
    return np.clip(np.floor((Q50 * s + 50.0) / 100.0), 1.0, 255.0)

def jpeg_like(img, quality):
    '''分块 DCT + 量化 + 反变换（JPEG 的核心损失环节，省略了色度下采样与熵编码）。'''
    x = np.clip(img, 0, 1) * 255.0 - 128.0
    H, W, C = x.shape
    assert H % 8 == 0 and W % 8 == 0, '为简化，要求尺寸是 8 的倍数'
    Q = qtable(quality)
    out = np.zeros_like(x)
    for c in range(C):
        for i in range(0, H, 8):
            for j in range(0, W, 8):
                blk = x[i:i + 8, j:j + 8, c]
                co = DCT8 @ blk @ DCT8.T
                co = np.round(co / Q) * Q
                out[i:i + 8, j:j + 8, c] = DCT8.T @ co @ DCT8
    return np.clip((out + 128.0) / 255.0, 0.0, 1.0)

def psnr(a, b):
    return float(10.0 * np.log10(1.0 / max(np.mean((a - b) ** 2), 1e-12)))

def patch_psnr(a, b, box):
    x0, y0, x1, y1 = box
    return psnr(a[y0:y1, x0:x1], b[y0:y1, x0:x1])

def hf_distortion(a, b, box):
    # 牌面高频被**改动**了多少（注意不是「衰减」：块效应与振铃会**凭空造出**高频）
    x0, y0, x1, y1 = box
    ha = np.diff(a[y0:y1, x0:x1].mean(-1), axis=1)
    hb = np.diff(b[y0:y1, x0:x1].mean(-1), axis=1)
    return float(np.linalg.norm(hb - ha) / (np.linalg.norm(ha) + 1e-12))

def blockiness(im):
    # 块效应指标：8 的倍数列上的梯度 / 其余列上的梯度
    p = im.mean(-1)
    gh = np.abs(np.diff(p, axis=1))
    on = ((np.arange(gh.shape[1]) + 1) % 8 == 0)
    return float(gh[:, on].mean() / (gh[:, ~on].mean() + 1e-12))

sc24 = make_scene('禁令(红)', Z=25.0)         # 24 px 的标志 —— 只占 9 个 8×8 块
n_blocks = int(np.ceil(sc24['d_px'] / 8)) ** 2
print(f'标志直径 {sc24["d_px"]:.0f} px → 整个牌面只占 **{n_blocks} 个 8×8 块**')
print()
print(f"{'quality':>8s} {'DC步长':>7s} {'全图PSNR':>10s} {'牌面PSNR':>10s} "
      f"{'牌面高频畸变':>13s} {'块效应':>8s} {'色相偏移':>9s}")
res, resp, hfd = {}, {}, {}
for q in [95, 85, 70, 50, 30, 15]:
    im = jpeg_like(sc24['img'], q)
    res[q] = psnr(sc24['img'], im)
    resp[q] = patch_psnr(sc24['img'], im, sc24['bbox'])
    hfd[q] = hf_distortion(sc24['img'], im, sc24['bbox'])
    print(f'{q:>8d} {qtable(q)[0,0]:>7.0f} {res[q]:>9.2f} dB {resp[q]:>9.2f} dB '
          f'{hfd[q]:>12.1%} {blockiness(im):>8.2f} '
          f'{hue_shift(sign_hue(sc24, im)[0], sign_hue(sc24)[0]):>8.2f}°')

assert res[95] > res[70] > res[30] > res[15], '全图 PSNR 必须随 quality 单调'
assert resp[95] > resp[70] > resp[30] > resp[15], '牌面 PSNR 必须随 quality 单调'
assert hfd[95] < hfd[70] < hfd[30] < hfd[15], '牌面高频畸变必须随 quality 下降而增大'
assert res[15] - resp[15] > 8.0, '牌面 PSNR 应当比全图 PSNR 差 8 dB 以上'
assert blockiness(jpeg_like(sc24['img'], 15)) > 1.5 * blockiness(sc24['img'])
print()
print(f'★ quality=15 时：全图 PSNR {res[15]:.1f} dB 看着还行，'
      f'**牌面 PSNR 只有 {resp[15]:.1f} dB**（差 {res[15]-resp[15]:.1f} dB）')
print('  —— 压缩损失**集中在标志上**，因为标志正是画面里高频最密的地方。')
print('  用全图 PSNR 评估压缩对 TSR 的影响，会把危害低估一个数量级。')
print()
print('⚠️  关键观察：**颜色（低频）几乎不掉，牌面高频被改得面目全非。**')
print('    注意「高频畸变」不等于「高频衰减」：块效应与振铃会**凭空造出**假边缘，')
print('    所以简单地看梯度能量反而会上升 —— 必须看与原图的差异，不是看绝对能量。')
print('    对应的故障指纹极有特征：')
print('    「检测召回正常（轮廓+颜色还在）、细粒度分类准确率异常低（60/80 分不清）」')
print('    —— 在两级架构里表现为「检测器没问题、分类器背锅」。')
print('✅ 而且训练与部署的编码链路通常不同：')
print('   训练 = 相机→JPEG存盘→解码→训练；车端 = 相机→ISP→直接送模型（无 JPEG）。')
print('   这是一种**反向域差**，必须在 C60 模块 01 的预处理对拍里一起查。')

## ✏️ 练习 1：从数据里推出安全的色相抖动上限

不要用我们写死的四个代表色，而是**从「训练集」统计**每个颜色族的色相分布，
再算出安全上限。实现两个函数：

- `hue_stats(samples)`：`samples = {类名: [色相数组]}` → `{类名: (圆均值, p05, p95)}`。
  **色相是圆周量，必须用圆均值**：`atan2(mean(sin h), mean(cos h))`。
- `safe_hue_limit(stats, safety=0.5)`：相邻类的间隔 = 圆均值之差；
  有效余量 = 半间隔 − 本类的半散布（`(p95-p05)/2`）；返回 `safety ×` 全局最小有效余量。

In [ ]:
def hue_stats(samples):
    # TODO: 对每个类算 (圆均值[0,360), p05, p95)
    #       圆均值：ang = atan2(mean(sin), mean(cos))，再转成度并对 360 取模
    #       分位数：先把角度对齐到圆均值附近（减去均值后 wrap 到 [-180,180]），再取分位数、加回
    raise NotImplementedError

def safe_hue_limit(stats, safety=0.5):
    # TODO: ① 按圆均值排序 ② 相邻（含首尾绕回）间隔的一半 = 几何余量
    #       ③ 有效余量 = 几何余量 − 本类半散布 (p95-p05)/2，两侧取较小者
    #       ④ 返回 safety × 全局最小有效余量（不小于 0）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
_r = np.random.default_rng(11)
SAMPLES = {}
for k, h in CLASS_HUE.items():
    spread = 6.0 if k == '指示(蓝)' else 4.0          # 类内色相散布（真实数据里一定存在）
    SAMPLES[k] = (h + _r.normal(0, spread, 4000)) % 360.0

st = hue_stats(SAMPLES)
for k in CLASS_HUE:
    mu, lo, hi = st[k]
    print(f'{k:<12s} 圆均值 {mu:7.2f}°（真值 {CLASS_HUE[k]:7.2f}°）  '
          f'p05={lo:7.2f}°  p95={hi:7.2f}°  半散布 {((hi-lo)%360)/2:5.2f}°')
    #                                             ↑ 红色会跨 0°，必须用圆周差
    assert hue_dist(mu, CLASS_HUE[k]) < 0.5, f'{k} 圆均值偏差过大'

lim = safe_hue_limit(st, safety=0.5)
lim_strict = safe_hue_limit(st, safety=0.25)
print(f'\n安全上限（safety=0.5）  = ±{lim:.2f}°')
print(f'安全上限（safety=0.25） = ±{lim_strict:.2f}°')
assert 5.0 < lim < 0.5 * MIN_MARGIN, f'扣掉类内散布后必须比理想余量 {0.5*MIN_MARGIN:.1f}° 更保守'
assert abs(lim_strict - lim / 2) < 1e-9
# 红色圆均值±上限 不能翻类
for sgn in (+1, -1):
    assert classify_by_hue((st['禁令(红)'][0] + sgn * lim) % 360.0) == '禁令(红)'
print('\n✅ 练习 1 通过：**幅度不是抄来的，是从数据里算出来的。**')
print(f'   注意它比「理想代表色」算出的 ±{0.5*MIN_MARGIN:.1f}° 更严 —— 类内散布已经吃掉了一部分余量。')

## ✏️ 练习 2：雾天的可用距离预算

实现 `fog_budget(V, t_min, speed, react_time)`，返回一个字典：

- `beta`：散射系数 `-ln(0.05)/V`
- `d_max`：保住 `t_min` 对比度的最远距离 `ln(1/t_min)/beta`
- `t_avail`：`d_max / speed`（秒）
- `ok`：`t_avail >= react_time` 才为 True
- `max_speed`：为了满足 `react_time`，允许的最高车速 `d_max / react_time`

In [ ]:
def fog_budget(V, t_min=0.20, speed=25.0, react_time=1.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
b = fog_budget(50.0, 0.20, 25.0, 1.5)
assert abs(b['beta'] - (-np.log(0.05) / 50.0)) < 1e-12
assert abs(b['d_max'] - max_range_at_contrast(50.0, 0.20)) < 1e-9
assert abs(b['t_avail'] - b['d_max'] / 25.0) < 1e-12
assert b['ok'] is False or b['ok'] == False, '50 m 能见度 + 90 km/h 必须判为不满足'
assert abs(b['max_speed'] - b['d_max'] / 1.5) < 1e-9

print(f"{'能见度':>8s} {'d_max':>8s} {'可用时间':>9s} {'满足1.5s?':>10s} {'建议限速':>12s}")
for V in [500, 200, 100, 50, 20]:
    r = fog_budget(V, 0.20, 25.0, 1.5)
    print(f'{V:>7.0f} m {r["d_max"]:>7.1f} m {r["t_avail"]:>8.2f} s '
          f'{("✅" if r["ok"] else "❌"):>9s} {r["max_speed"]*3.6:>9.0f} km/h')

assert fog_budget(500.0)['ok'] and not fog_budget(50.0)['ok']
assert fog_budget(200.0, speed=25.0)['t_avail'] > fog_budget(100.0, speed=25.0)['t_avail']
print('\n✅ 练习 2 通过：能见度直接换算成**决策时间预算**。')
print('   这个预算表就是「雾天该不该降级/降速」的量化依据 ——')
print('   比「模型在雾天 mAP 掉了 12 个点」有用得多，因为它能直接连到控制策略。')

## ✏️ 练习 3：曝光时间预算（模糊 vs 噪声的二选一）

实现 `exposure_budget(X, v, S, Z, ratio_max)`：给定可接受的 `L_blur/s_px` 上限，
反解允许的最长曝光时间 `t_max = ratio_max·S·Z/(X·v)`；
再实现 `night_tradeoff(...)` 返回 `{'t_max':…, 'gain_needed':…, 'snr_ratio':…}`：
若目标曝光 `t_target` 超过 `t_max`，就必须用数字增益补上缺口
`gain = t_target/t_max`，代价是 SNR 降为 `1/√gain`。

In [ ]:
def exposure_budget(X, v, S, Z, ratio_max=0.15):
    # TODO: 返回允许的最长曝光时间（秒）
    raise NotImplementedError

def night_tradeoff(X, v, S, Z, t_target, ratio_max=0.15):
    # TODO: 返回 {'t_max':…, 'gain_needed':…, 'snr_ratio':…}
    #       gain_needed = max(1.0, t_target / t_max)；snr_ratio = 1/sqrt(gain_needed)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
tm = exposure_budget(3.0, 25.0, 0.6, 10.0, 0.15)
assert abs(tm - 0.15 * 0.6 * 10.0 / (3.0 * 25.0)) < 1e-15, tm
assert abs(tm - 0.012) < 1e-12
# 与第 5 节的正向公式对拍
assert abs(blur_len_px(F_PX, 3.0, 10.0, 25.0, tm) / sign_px(F_PX, 0.6, 10.0) - 0.15) < 1e-12

print(f"{'场景':<16s} {'Z(m)':>6s} {'v(m/s)':>7s} {'t_max':>9s} {'目标曝光':>9s} "
      f"{'需增益':>8s} {'SNR 变为':>9s}")
for name, Z_, v_, tt in [('白天巡航', 30, 25, 0.005), ('白天近距', 10, 25, 0.005),
                         ('夜间城区', 10, 14, 0.030), ('夜间高速', 10, 25, 0.030),
                         ('雨夜',     8, 20, 0.040)]:
    r = night_tradeoff(3.0, v_, 0.6, Z_, tt, 0.15)
    print(f'{name:<16s} {Z_:>6d} {v_:>7d} {r["t_max"]*1000:>7.1f} ms {tt*1000:>7.0f} ms '
          f'{r["gain_needed"]:>8.2f}× {r["snr_ratio"]:>8.2f}×')

r_day = night_tradeoff(3.0, 25.0, 0.6, 30.0, 0.005)
r_night = night_tradeoff(3.0, 25.0, 0.6, 10.0, 0.030)
assert abs(r_day['gain_needed'] - 1.0) < 1e-12, '白天巡航不需要额外增益'
assert r_night['gain_needed'] > 2.0 and r_night['snr_ratio'] < 0.75
assert abs(r_night['snr_ratio'] - 1 / np.sqrt(r_night['gain_needed'])) < 1e-12
print('\n✅ 练习 3 通过：夜间是一个**没有免费午餐的二选一**。')
print('   要么模糊（拉长曝光），要么噪声（拉高增益），两条路削掉的是同一样东西：')
print('   **牌面上区分「限速 60」和「限速 80」的那点高频。**')
print('   所以「夜间为什么掉点」的完整回答必须同时提到这两条路。')

## ✏️ 练习 4：光度增强配置审计器

实现 `audit_photometric(cfg, hue_limit)`，输入是一份增强配置
（`{算子名: {'p':…, 'amp':…}}`），返回 `{'errors': [...], 'warnings': [...], 'ok': bool}`：

1. **error**：出现禁用算子（`channel_shuffle` / `to_gray` / `invert`），且 `p > 0`；
2. **error**：`hue` 的 `amp` 超过 `hue_limit`；
3. **error**：配置里出现 `split == 'val'` 却带随机算子（`cfg` 里给了 `'_split'` 字段）；
4. **warning**：`low_light` 的 `p > 0` 但 `motion_blur` 的 `p == 0`（只模拟了高增益那条路）；
5. **warning**：所有算子概率之积对应的「全触发率」`< 0.001`（增强名义上很强、实际几乎不发生）。

In [ ]:
BANNED = {'channel_shuffle', 'to_gray', 'invert'}

def audit_photometric(cfg, hue_limit):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
GOOD = {'_split': 'train',
        'brightness_mul': {'p': 0.8, 'amp': 0.35}, 'contrast': {'p': 0.5, 'amp': 0.25},
        'gamma': {'p': 0.4, 'amp': 0.35}, 'saturation': {'p': 0.6, 'amp': 0.4},
        'hue': {'p': 0.3, 'amp': 12.0}, 'low_light': {'p': 0.25, 'amp': 1.0},
        'motion_blur': {'p': 0.25, 'amp': 1.0}, 'fog': {'p': 0.15, 'amp': 1.0}}
r = audit_photometric(GOOD, hue_limit=12.6)
print('GOOD :', r)
assert r['ok'] and not r['errors'], r

BAD = dict(GOOD)
BAD['hue'] = {'p': 0.5, 'amp': 40.0}                 # albumentations 默认
BAD['channel_shuffle'] = {'p': 0.1, 'amp': 1.0}
r2 = audit_photometric(BAD, hue_limit=12.6)
print('BAD  :', r2)
assert not r2['ok'] and len(r2['errors']) >= 2
assert any('hue' in e for e in r2['errors']) and any('channel_shuffle' in e for e in r2['errors'])

VAL = {'_split': 'val', 'brightness_mul': {'p': 0.5, 'amp': 0.2}}
r3 = audit_photometric(VAL, hue_limit=12.6)
print('VAL  :', r3)
assert not r3['ok'] and any('val' in e for e in r3['errors']), '验证集出现随机算子必须报错'

ONLY_NOISE = {'_split': 'train', 'low_light': {'p': 0.3, 'amp': 1.0},
              'motion_blur': {'p': 0.0, 'amp': 1.0}}
r4 = audit_photometric(ONLY_NOISE, hue_limit=12.6)
print('NOISE:', r4)
assert any('motion_blur' in w for w in r4['warnings']), '只加噪不加模糊应当告警'

WEAK = {'_split': 'train', **{f'op{i}': {'p': 0.2, 'amp': 0.1} for i in range(5)}}
r5 = audit_photometric(WEAK, hue_limit=12.6)
assert any('触发' in w for w in r5['warnings']), '概率相乘过低应当告警'
print('WEAK :', r5)
print('\n✅ 练习 4 通过：把「配方纪律」写成代码，比写进文档可靠得多。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def hue_stats(samples):
    out = {}
    for k, arr in samples.items():
        a = np.asarray(arr, dtype=float) % 360.0
        rad = np.deg2rad(a)
        mu = np.rad2deg(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360.0
        rel = (a - mu + 180.0) % 360.0 - 180.0            # 对齐到均值附近再取分位数
        lo = (mu + np.percentile(rel, 5)) % 360.0
        hi = (mu + np.percentile(rel, 95)) % 360.0
        out[k] = (float(mu), float(lo), float(hi))
    return out

def safe_hue_limit(stats, safety=0.5):
    names = sorted(stats, key=lambda k: stats[k][0])
    mus = [stats[k][0] for k in names]
    n = len(names)
    best = float('inf')
    for i, k in enumerate(names):
        gap_next = (mus[(i + 1) % n] - mus[i]) % 360.0
        gap_prev = (mus[i] - mus[(i - 1) % n]) % 360.0
        lo, hi = stats[k][1], stats[k][2]
        half_spread = ((hi - lo) % 360.0) / 2.0          # 红色跨 0°，必须用圆周差
        eff = min(gap_next, gap_prev) / 2.0 - half_spread
        best = min(best, eff)
    return float(max(0.0, safety * best))

In [ ]:
# 练习 2 参考答案
def fog_budget(V, t_min=0.20, speed=25.0, react_time=1.5):
    beta = -np.log(0.05) / float(V)
    d_max = float(np.log(1.0 / t_min) / beta)
    t_avail = d_max / float(speed)
    return {'beta': float(beta), 'd_max': d_max, 't_avail': float(t_avail),
            'ok': bool(t_avail >= react_time), 'max_speed': float(d_max / react_time)}

In [ ]:
# 练习 3 参考答案
def exposure_budget(X, v, S, Z, ratio_max=0.15):
    return float(ratio_max * S * Z / (X * v))

def night_tradeoff(X, v, S, Z, t_target, ratio_max=0.15):
    t_max = exposure_budget(X, v, S, Z, ratio_max)
    gain = max(1.0, float(t_target) / t_max)
    return {'t_max': t_max, 'gain_needed': gain, 'snr_ratio': float(1.0 / np.sqrt(gain))}

In [ ]:
# 练习 4 参考答案
BANNED = {'channel_shuffle', 'to_gray', 'invert'}      # 成像链路上无对应 → TSR 禁用

def audit_photometric(cfg, hue_limit):
    errors, warnings = [], []
    split = cfg.get('_split', 'train')
    ops = {k: v for k, v in cfg.items() if not k.startswith('_')}
    for name, spec in ops.items():
        p = float(spec.get('p', 0.0))
        if name in BANNED and p > 0:
            errors.append(f'禁用算子 {name} 出现且 p={p}（成像链路上没有对应，等价于改标签）')
        if name == 'hue' and float(spec.get('amp', 0.0)) > hue_limit:
            errors.append(f'hue 幅度 {spec["amp"]}° > 安全上限 {hue_limit}°（会跨越颜色族边界）')
        if split == 'val' and p > 0:
            errors.append(f'val 集出现随机算子 {name}（验证集只允许确定性变换）')
    if ops.get('low_light', {}).get('p', 0) > 0 and ops.get('motion_blur', {}).get('p', 0) == 0:
        warnings.append('low_light 开启但 motion_blur 关闭：只模拟了高增益那条路，'
                        '缺了长曝光那条路')
    if ops:
        all_fire = float(np.prod([float(s.get('p', 0.0)) for s in ops.values()]))
        if all_fire < 1e-3:
            warnings.append(f'全部算子同时触发的概率仅 {all_fire:.2e} —— '
                            f'配方名义上很强，实际强度远低于直觉')
    return {'errors': errors, 'warnings': warnings, 'ok': not errors}

---
## 🧪 真实工程胶囊：一份可直接抄的 TSR 光度增强配置与验证脚本

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
#  TSR 光度增强配方（albumentations 风格）—— 每一行都能说出反推自哪个失效模式
# ══════════════════════════════════════════════════════════════════════
import albumentations as A

HUE_LIMIT_CV = 6      # ★ OpenCV 单位！1 单位 = 2°，所以 6 → ±12°
                      #   默认值 20 等于 ±40°，会把红色禁令牌推进黄色警告牌的色相区间
                      #   （红→黄的边界只有约 25°，见本 notebook 第 3 节）

train_tf = A.Compose([
    # ── 顺序遵循成像链路：几何 → 大气 → 曝光/ISP → 光学 → 传感器 → 编码 ──
    # 几何增强见模块 01（注意：交通标志**禁止水平翻转**）
    A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.35, p=0.15),      # 近似；带深度的版本见下
    A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.25, p=0.8),
    A.RandomGamma(gamma_limit=(75, 140), p=0.4),
    A.HueSaturationValue(hue_shift_limit=HUE_LIMIT_CV,                  # ★ 唯一必须捏死的旋钮
                         sat_shift_limit=35, val_shift_limit=25, p=0.4),
    A.MotionBlur(blur_limit=(3, 7), p=0.25),                            # 长度按 X·v·t/(S·Z) 定
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.25),
    A.ImageCompression(quality_lower=55, quality_upper=95, p=0.2),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'],
                            min_visibility=0.25))

# ★ 验证集：一个随机算子都不能有（铁律）
val_tf = A.Compose([A.NoOp()], bbox_params=A.BboxParams(format='pascal_voc',
                                                        label_fields=['labels']))

# ── 带深度的物理雾化（albumentations 的 RandomFog 与深度无关，只能近似）──
def physical_fog(img, depth_m, visibility_m, A_light=None, rng=None):
    # I = J*t + A*(1-t), t = exp(-beta*d), beta = -ln(0.05)/V
    rng = rng or np.random.default_rng()
    beta = -np.log(0.05) / float(visibility_m)
    t = np.exp(-beta * depth_m)[..., None]
    if A_light is None:                       # ★ 随机化 A，避免「合成指纹」
        A_light = rng.uniform(0.75, 0.95) * np.array([1.0, 1.01, 1.06])
    return np.clip(img * t + np.asarray(A_light).reshape(1, 1, 3) * (1 - t), 0, 1)

# ── 物理正确的低光合成（img + N(0,sigma) 是错的）──
def physical_low_light(img, exposure_ratio, full_well=8000., read_e=3., gamma=2.2, rng=None):
    rng = rng or np.random.default_rng()
    lin = np.clip(img, 0, 1) ** gamma                       # sRGB -> 线性
    N = lin * full_well * exposure_ratio                    # 光电子数
    sig = rng.poisson(N) + rng.normal(0, read_e, N.shape)   # 散粒 + 读出
    out = np.clip(sig / (full_well * exposure_ratio), 0, 1) ** (1 / gamma)   # 数字增益 + 编码
    return np.round(out * 255) / 255                        # 8-bit 量化

# ══════════════════════════════════════════════════════════════════════
#  上线前的四项检查（缺一项都可能让「增强」变成「加标签噪声」）
# ══════════════════════════════════════════════════════════════════════
# ① 色相红线：从训练集统计每个颜色族的色相分位数，算出真实类间余量，
#    再取 0.5x 作为 hue_shift_limit。**不要抄默认值。**
# ② 单位换算：albumentations/OpenCV 的 hue 单位是 0-179（1 单位 = 2 度）；
#    torchvision ColorJitter 的 hue 单位是「圈」（0.1 = 36 度）。
# ③ 分桶验证：整体 mAP 会掩盖颜色族错分。必须看
#    (a) 跨颜色族混淆矩阵  (b) 夜间/雾天切片 AP  (c) 细粒度分类准确率（限速 60 vs 80）
# ④ 验证集零随机：断言 val_tf 里不含任何 p<1 的随机算子（写成 CI 检查）
#
# ── 反推表：每个算子对应哪个失效模式（配方 review 时逐行对照）──
#   brightness/contrast  <- 隧道出入口、逆光、阴影
#   gamma                <- 不同车型 ISP 的色调映射曲线差异
#   saturation           <- 褪色标志、雾天去饱和
#   hue (小幅)           <- 白平衡漂移（真实幅度就这么小）
#   low_light + blur     <- 夜间：拉增益 or 拉曝光，二选一，必须成对出现
#   fog (带深度)         <- 雨雾雪天召回塌陷
#   ImageCompression     <- 回传链路压缩；限速数字对分不清
#   ❌ channel_shuffle / to_gray / invert  <- 成像链路上无对应，**禁用**
'''
print(RECIPE)
for token in ['HUE_LIMIT_CV', '1 单位 = 2°', 'physical_fog', 'physical_low_light',
              'min_visibility', 'val_tf', '禁用', 'exp(-beta*d)']:
    assert token in RECIPE, token
print('✅ 配方覆盖：色相红线 / 单位换算 / 物理雾化 / 物理低光 / 验证集零随机 / 失效模式反推表')

### 小结

- **光度增强不改一个标注，却可能改掉标签。** 「不用同步改标注」制造了它很安全的错觉；
  真正的判据是 `p(y | T(x)) = p(y | x)` —— 而 TSR 里颜色进入了 `p(y|x)`。
- **色相就是语义，红线是 25.2°。** 用一组代表色，红色禁令牌正向偏移约 25° 就落进黄色警告的
  色相区间；`albumentations` 默认 `hue_shift_limit=20`（OpenCV 单位 = **±40°**）直接越界，
  `torchvision ColorJitter(hue=0.1)` = ±36° 同样越界；YOLOv5 的 `hsv_h=0.015` 是乘性的，
  只有 ±5.4°，反而安全。**建议上限 = 0.5 × 从数据统计出的最小有效余量。**
- **色彩代数一句话**：亮度/对比度/雾都是逐通道仿射 `aI+b` → **色相严格不变**（b 中性时），
  **`b>0` 必然掉饱和度**；gamma 非线性但只移约 1.4°；显式色相抖动是唯一能跨类别边界的算子。
- **雾要用物理模型**：`I = J·t + A(1-t)`，`t = e^(-βd)`，`β ≈ 3/V`。对比度被严格乘以 `t`，
  饱和度约按 `t` 衰减，色相偏移**全部来自 A 的非中性**。
  能见度 50 m 时标志最远只能在 **27 m** 保住 20% 对比度 → 90 km/h 下只剩 **1.1 秒**。
- **夜间是没有免费午餐的二选一**：拉曝光 → 模糊按 `t_exp` 线性增长；拉增益 → SNR 按 `√N` 下降。
  两条路削的是同一样东西（区分 60/80 的高频）。**所以低光合成与运动模糊必须成对进配方。**
- **`L_blur/s_px = X·v·t_exp/(S·Z)` 与焦距无关** —— 上长焦解决不了运动模糊。
- **卷帘畸变不用做成增强**（小目标帧内错切 < 1 px），但**时序融合必须用行时间戳**：
  20 ms 读出时间在 30 m/s 下是 0.6 m 的自车位移，足以让 IoU 关联失败。
- **JPEG 的 8×8 块对小标志是灾难**：24 px 的牌面只占 9 个块，颜色（低频）几乎不掉但
  牌面高频掉得很快 → 故障指纹是「检测正常、细粒度分类崩」。
- **增强算子必须能挂到成像链路的某一环上**；挂不上去的（channel shuffle / 灰度化 / 反色）
  在 TSR 上等价于随机改标签。

下一站：**模块 03 · 混合类增强** —— Mosaic / MixUp / CutMix / Copy-Paste，
以及长尾类别最直接的那个解法。